# Тестовое задание Антибот-классификатор

## 1. Импорт библиотек

In [134]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

from metric import precision_at_recall

SEED = 42
EVENT_TYPES = [
    "search_results_view", "item_view", "photo_swipe", "seller_page_view",
    "contact_phone_show", "contact_chat_open", "contact_message_sent",
    "favorite_add", "login", "captcha_shown",
]

## 2. Загрузка данных

Объединяем train и test, чтобы признаки для них считались одинаково, можем это сделать, потому что наш target, который предсказываем, не используется в признаках. `row_id` это номер строки, т.к. один `cookie_id` может встретиться в разных суточных окнах.

In [135]:
date_columns = ["cookie_created_at", "window_start_ts", "window_end_ts"]

train = pd.read_csv("data/train.csv", parse_dates=date_columns)
test = pd.read_csv("data/test.csv", parse_dates=date_columns)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

combined = pd.concat([train, test], ignore_index=True) 
combined_with_id = combined.reset_index(names="row_id")
print("train:", train.shape, "test:", test.shape, "events:", events.shape)
'''
#чтобы посмотреть струтуру файла
display(train.head())
display(test.head())
display(events.head())
'''

train: (11091, 5) test: (4909, 4) events: (328905, 14)


'\n#чтобы посмотреть струтуру файла\ndisplay(train.head())\ndisplay(test.head())\ndisplay(events.head())\n'

## 3. События внутри окна

Привязываем `events` к строкам train/test по `cookie_id`, затем оставляем только события внутри соответствующего окна. Дальше группируем по `row_id`, чтобы разные окна одной cookie не смешивались.

In [136]:
window_events = events.merge(
    combined_with_id[["row_id", "cookie_id", "window_start_ts", "window_end_ts"]],
    on="cookie_id",
)

# проверяем на то, что событие внутри окна
inside_window = (
    (window_events.event_ts >= window_events.window_start_ts)
    & (window_events.event_ts < window_events.window_end_ts)
)
window_events = window_events.loc[inside_window].copy()

# сортируем и по окну, и по времени, что потом можно было без проблем работать с вычислением пауз и т.д.
window_events = window_events.sort_values(["row_id", "event_ts"], kind="stable")

groups = window_events.groupby("row_id", sort=False)

## 4. Простые признаки

Для каждой строки считаем возраст cookie, число событий, длительность активности и разнообразие действий. (отдельно прокомментировал неочевидные признаки)

In [137]:
features = pd.DataFrame(index=combined.index)

# возраст cookie
features["cookie_age_days"] = (
    (combined.window_start_ts - combined.cookie_created_at) / pd.Timedelta(days=1)
).to_numpy()

# куки появившиеся внутри окна, оставляет True или False
features["created_in_window"] = (
    combined.cookie_created_at >= combined.window_start_ts
).astype(int)

# количество ивентов и секунд в сессиях
features["n_events"] = groups.size()
features["active_seconds"] = (groups.event_ts.max() - groups.event_ts.min()).dt.total_seconds()

features["first_event_hour"] = (
    groups.event_ts.min() - groups.window_start_ts.first()
) / pd.Timedelta(hours=1)
features["last_event_hour"] = (
    groups.event_ts.max() - groups.window_start_ts.first()
) / pd.Timedelta(hours=1)

# в какое различное время был активен пользователь
features["n_active_hours"] = groups.event_ts.apply(lambda times: times.dt.hour.nunique())

# всякие уникальные фичи, по типу категорий, локаций, юзер-агентов и тд.
features["n_items"] = groups.item_id.nunique()
features["n_categories"] = groups.item_category.nunique()
features["n_locations"] = groups.item_location.nunique()
features["n_queries"] = groups.search_query.nunique()
features["n_agents"] = groups.user_agent.nunique()
features["n_platforms"] = groups.platform.nunique()

# в целом смотрим сколько событий пришлось на каждую категорию внутри окна
category_counts = window_events.groupby(["row_id", "item_category"]).size()

# процент кол-ва максимально частой категории относительно других
features["max_category_share"] = (
    category_counts.groupby(level=0).max() / groups.item_category.count()
)

# максимальное кол-во просмотров 1-го товара
item_views = window_events.loc[window_events.event_name == "item_view"]
item_view_counts = item_views.groupby(["row_id", "item_id"]).size()
features["max_item_views"] = item_view_counts.groupby(level=0).max()

## 5. Количество и доли типов событий

Считаем, сколько раз был каждый тип действия, и добавляем простые отношения между ними.

In [138]:
# строим таблицу смежности в каждом id с ивентами
event_counts = pd.crosstab(window_events.row_id, window_events.event_name)
event_counts = event_counts.reindex(columns=EVENT_TYPES, fill_value=0)
#display(event_counts)

# обьединяем 2 датафрейма, префикс я добавил, чтобы не перепутать столбцы
features = features.join(event_counts.add_prefix("n_"))

# доля конкретных ивентов от общего числа событий
for event_name in EVENT_TYPES:
    features["share_" + event_name] = features["n_" + event_name] / features.n_events
#display(features)

features["items_per_view"] = features.n_items / features.n_item_view.clip(lower=1)
features["queries_per_search"] = features.n_queries / features.n_search_results_view.clip(lower=1)
features["views_per_search"] = features.n_item_view / features.n_search_results_view.clip(lower=1)
features["contacts_per_view"] = (
    features.n_contact_phone_show + features.n_contact_chat_open + features.n_contact_message_sent
) / features.n_item_view.clip(lower=1)

## 6. Gaps и регулярность действий

Разница между соседними событиями помогает заметить регулярное автоматическое поведение.

In [139]:
window_events["gap"] = groups.event_ts.diff().dt.total_seconds()
gaps = window_events.groupby("row_id").gap

features["gap_mean"] = gaps.mean()
features["gap_median"] = gaps.median()
features["gap_std"] = gaps.std()
features["gap_min"] = gaps.min()
features["gap_max"] = gaps.max()

# (!) тут мы измеряем долю гэпов между ивентами, которые не превысили наши пороги
# это помогает понять, как быстро пользователь (или бот) совершает действия
for seconds in [1, 5, 30, 300, 3600]:
    features[f"gap_le_{seconds}"] = window_events.gap.le(seconds).groupby(window_events.row_id).mean()

# Коэффициент вариации гэпов (смотрим на регулярность действий)
features["gap_cv"] = features.gap_std / features.gap_mean.clip(lower=1)

features["gap_p10"] = gaps.quantile(0.1)
features["gap_p90"] = gaps.quantile(0.9)

# количество новых сессий
features["n_sessions"] = window_events.gap.gt(1800).groupby(window_events.row_id).sum() + 1

hour_counts = window_events.groupby(["row_id", window_events.event_ts.dt.floor("h")]).size()
features["max_events_hour"] = hour_counts.groupby(level=0).max()
features["events_hour_std"] = hour_counts.groupby(level=0).std()

# паузы между одиннаковыми событиями (по просмотрам и поискам)
for event_name in ["item_view", "search_results_view"]:
    same_events = window_events.loc[
        window_events.event_name == event_name, ["row_id", "event_ts"]
    ].copy()
    
    same_events["same_gap"] = same_events.groupby("row_id").event_ts.diff().dt.total_seconds()
    same_gaps = same_events.groupby("row_id").same_gap
    
    features[f"{event_name}_gap_median"] = same_gaps.median()
    features[f"{event_name}_gap_std"] = same_gaps.std()
    
    # доля одиннаковых событий, происходящих с интервалом <= 30 секунд
    features[f"{event_name}_gap_le_30"] = (
        same_events.same_gap.le(30).groupby(same_events.row_id).mean()
    )

## 7. Нормализация

Платформы приводим к общим названиям. Из User-Agent берём только простые текстовые признаки.

In [140]:
window_events["platform_norm"] = window_events.platform.str.lower().replace({
    "desktop": "web", "iphone": "ios",
})

# считаем в новую фичу долю событий с каждой платформы
for platform in ["web", "android", "ios"]:
    features["platform_" + platform] = (
        window_events.platform_norm.eq(platform).groupby(window_events.row_id).mean()
    )

user_agents = window_events.user_agent.fillna("").str.lower()

# выписал сюда самые странные запросы по моему мнению :)
agent_words = [
    "headless", "python", "bot", "curl", "selenium", "playwright",
    "firefox", "yabrowser", "mobile", "linux",
]

# доля таких запросов
for word in agent_words:
    features["ua_" + word] = (
        user_agents.str.contains(word, regex=False).groupby(window_events.row_id).mean()
    )

## 8. Поиск, координаты и переходы

Считаем глубину поиска, заполненность и значения координат, а также несколько пар соседних действий.

In [141]:
for column in ["search_page", "pointer_x", "pointer_y"]:
    # пытаемся заполнить числами и считаем их долю
    values = pd.to_numeric(window_events[column], errors="coerce")
    features[column + "_present"] = values.notna().groupby(window_events.row_id).mean()
    
    grouped_values = values.groupby(window_events.row_id)
    features[column + "_mean"] = grouped_values.mean()
    features[column + "_std"] = grouped_values.std()
    features[column + "_max"] = grouped_values.max()
    
    if column == "search_page":
        for page in [2, 5, 10]:
            features[f"search_page_ge_{page}"] = values.ge(page).groupby(window_events.row_id).sum()
    else:
        features[column + "_nunique"] = values.groupby(window_events.row_id).nunique()

query_counts = window_events.groupby(["row_id", "search_query"]).size()
features["max_query_repeats"] = query_counts.groupby(level=0).max()

max_pages = window_events.groupby(["row_id", "search_query"]).search_page.max()
features["max_search_page_per_query"] = max_pages.groupby(level=0).max()

# смотрим на события перед текущим
window_events["prev_event"] = groups.event_name.shift()

# а тут подозрительные переходы, которые присущи ботам
transitions = [
    ("search_results_view", "item_view"),
    ("item_view", "item_view"),
    ("item_view", "contact_phone_show"),
    ("item_view", "photo_swipe"),
    ("captcha_shown", "item_view"),
]
# и тут считаем именно количество таких переходов
for before, after in transitions:
    matched = window_events.prev_event.eq(before) & window_events.event_name.eq(after)
    features[f"transition_{before}_{after}"] = matched.groupby(window_events.row_id).sum()

## 9. Два экстра признака коротких пауз и подготовка таблиц

Мы берем только паузы от 0 до 300 секунд. Логика такая: среднее покажет общий темп действий, а стандартное отклонение покажет, насколько человека "бросает" из стороны в сторону. Это только гипотеза, мы не утверждаем, что любое регулярное действие это стопроцентный бот. Дальше просто заменяем NaN'ы нулями и делим данные на train и test.

In [142]:
# долгие перерывы не включаем в расчёт регулярности быстрых действий.
short_gaps = window_events.gap.where(window_events.gap <= 300)

features["short_gap_mean"] = short_gaps.groupby(window_events.row_id).mean()
features["short_gap_std"] = short_gaps.groupby(window_events.row_id).std()

# очистка
features = features.replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
x_train = features.iloc[:len(train)]
x_test = features.iloc[len(train):]
y = train.target.to_numpy()

# проверка, что все хорошо
assert features.short_gap_mean.between(0, 300).all()
assert len(x_train) == len(train) == len(y)
assert len(x_test) == len(test)
print("Признаков:", len(features.columns))

Признаков: 100


## 10. Проверка на будущих датах

Обучаемся только на датах до начала проверки. Сначала оцениваем целую неделю с одним общим порогом, затем три прежних отрезка.


In [143]:
models = [
    HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.045, max_leaf_nodes=15,
        l2_regularization=3, random_state=SEED,
    ),
    HistGradientBoostingClassifier(
        max_iter=500, learning_rate=0.04, max_leaf_nodes=15,
        l2_regularization=3, random_state=SEED,
    ),
]

periods = [
    ("2026-04-13", "2026-04-20"),  # пересекается с проверками ниже
    ("2026-04-12", "2026-04-14"),
    ("2026-04-14", "2026-04-17"),
    ("2026-04-17", "2026-04-20"),
]
for start_date, end_date in periods:
    
    train_rows = train.window_start_ts.lt(start_date).to_numpy()
    valid_rows = (
        train.window_start_ts.ge(start_date) & train.window_start_ts.lt(end_date)
    ).to_numpy()
    
    scores = []
    for model in models:
        model.fit(x_train.iloc[train_rows], y[train_rows])
        scores.append(model.predict_proba(x_train.iloc[valid_rows])[:, 1])
    
    # наш ансамбль 75% 1 модели и 25% второй, можно сказать "контрольной"
    predictions = 0.75 * scores[0] + 0.25 * scores[1]
    print(
        f"{start_date}..{end_date}: "
        f"P@R>=0.7={precision_at_recall(y[valid_rows], predictions):.4f}, "
        f"PR-AUC={average_precision_score(y[valid_rows], predictions):.4f}, "
        f"ROC-AUC={roc_auc_score(y[valid_rows], predictions):.4f}"
    )

2026-04-13..2026-04-20: P@R>=0.7=0.7079, PR-AUC=0.7684, ROC-AUC=0.9332
2026-04-12..2026-04-14: P@R>=0.7=0.7500, PR-AUC=0.7772, ROC-AUC=0.9300
2026-04-14..2026-04-17: P@R>=0.7=0.6900, PR-AUC=0.7689, ROC-AUC=0.9323
2026-04-17..2026-04-20: P@R>=0.7=0.8550, PR-AUC=0.7925, ROC-AUC=0.9418


## 11. Финал

Берём 75% прогноза первой и 25% второй из ансамбля.


In [ ]:
scores = []
for model in models:
    model.fit(x_train, y)
    scores.append(model.predict_proba(x_test)[:, 1])

predictions = 0.75 * scores[0] + 0.25 * scores[1]
submission = pd.DataFrame({
    "cookie_id": test.cookie_id,
    "score": predictions,
})

assert submission.cookie_id.is_unique and submission.cookie_id.equals(test.cookie_id)
assert submission.score.notna().all() and submission.score.between(0, 1).all()

submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv with {len(submission)} rows ({len(features.columns)} features)")

Saved submission.csv with 4909 rows (100 features)
